# Prism CUDA Validation (H100)
**Runtime → Change runtime type → A100 or H100**

Compares orthogonal baseline vs Prism on GPT-2 small (124M) at real scale.
- Batch 64, seq_len 1024, 2000 steps
- ~20 min per run on H100, ~40 min total

In [ ]:
# Cell 1: Setup + Clone + Install (run this first)
import os, subprocess
if os.path.exists('/content/prism'):
    subprocess.run(['rm', '-rf', '/content/prism'])
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism
!pip install -q transformers datasets scipy
import torch
gpu = torch.cuda.get_device_name(0)
print(f'GPU: {gpu}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Ready.')

In [ ]:
# Cell 2: Run BOTH experiments (ortho baseline + Prism)
import sys, os, json, gc, time, warnings
import torch, numpy as np
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn

install_signal_handlers()
device = 'cuda'
os.makedirs('prism/results/cuda', exist_ok=True)

STEPS = 2000
EVALS = [250, 500, 750, 1000, 1500, 2000]

base_config = dict(
    max_steps=STEPS,
    eval_steps=EVALS,
    warmup_steps=200,
    log_every=50,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,  # effective batch 64
    max_length=1024,
    memory_pressure_threshold=5,
)

results = {}

# === RUN 1: Orthogonal baseline ===
print('='*60)
print('  RUN 1: Orthogonal baseline (lr=6.25e-5)')
print('='*60)
config_ortho = TrainConfig(**base_config, lr=6.25e-5)
t0 = time.time()
result_ortho = train(config_ortho, init_fn=make_init_fn('orthogonal'),
                     init_name='ortho', verbose=True)
results['ortho'] = result_ortho
print(f'\nOrtho done in {time.time()-t0:.0f}s — Final PPL: {result_ortho["final_ppl"]:.1f}')
_clear_memory(device); gc.collect()

# === RUN 2: Prism (Spectral Imprint + EigenTransfer + 2x LR) ===
print('\n' + '='*60)
print('  RUN 2: Prism (spectral + UV align + 2x LR)')
print('='*60)

with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)
print('Extracting pretrained directions for EigenTransfer...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')
init_fn = make_hybrid_init_fn(
    extracted['spectra_coeffs'], dirs,
    lam=1.0, align_mode='UV', align_strength=0.5
)
gc.collect()

config_prism = TrainConfig(**base_config, lr=1.25e-4)  # 2x LR
t0 = time.time()
result_prism = train(config_prism, init_fn=init_fn,
                     init_name='prism', verbose=True)
results['prism'] = result_prism
print(f'\nPrism done in {time.time()-t0:.0f}s — Final PPL: {result_prism["final_ppl"]:.1f}')

# === COMPARISON ===
print('\n' + '='*60)
print('  RESULTS')
print('='*60)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Config: GPT-2 small (124M), batch 64, seq_len 1024, {STEPS} steps')
print(f'\n{"Step":>6s}  {"Ortho":>8s}  {"Prism":>8s}  {"Ratio":>7s}')
print('-' * 35)
ortho_cp = result_ortho['checkpoints']
prism_cp = result_prism['checkpoints']
for step in EVALS:
    o = ortho_cp.get(step, None)
    p = prism_cp.get(step, None)
    if o and p:
        print(f'{step:>6d}  {o:>8.1f}  {p:>8.1f}  {o/p:>6.2f}x')
print(f'\nFinal: ortho={result_ortho["final_ppl"]:.1f}  prism={result_prism["final_ppl"]:.1f}  ratio={result_ortho["final_ppl"]/result_prism["final_ppl"]:.2f}x')
r = result_ortho['final_ppl'] / result_prism['final_ppl']
if r > 2.0: print('>>> PRISM ADVANTAGE HOLDS AT SCALE')
elif r > 1.3: print('>>> Prism advantage persists (reduced)')
else: print('>>> Prism advantage washes out')

In [ ]:
# Cell 3: Save + Download
from google.colab import files
out = {
    'gpu': torch.cuda.get_device_name(0),
    'steps': STEPS,
    'batch': 64,
    'seq_len': 1024,
    'ortho': {'final_ppl': result_ortho['final_ppl'], 'checkpoints': {str(k):v for k,v in result_ortho['checkpoints'].items()}, 'elapsed': result_ortho['elapsed']},
    'prism': {'final_ppl': result_prism['final_ppl'], 'checkpoints': {str(k):v for k,v in result_prism['checkpoints'].items()}, 'elapsed': result_prism['elapsed']},
}
with open('/content/cuda_results.json', 'w') as f:
    json.dump(out, f, indent=2)
files.download('/content/cuda_results.json')
print('Downloaded.')